# Spatio-Temporal Outbreak & Crisis Detection: Professional Data Ingestion

In a production-grade system, data ingestion goes far beyond simply reading a CSV. We must build a resilient pipeline. This notebook implements the core pillars of professional data engineering alongside standard ML preprocessing:

1. **Multi-Source Ingestion & API Polling**: Exponential backoff and rate limiting.
2. **Temporal Normalization**: Timezone standardization to UTC and uniform bucketing.
3. **Spatial Deduplication & Coordinate Validation**: Boundary checking and removing duplicate sensor signals.
4. **Handling Missing Data (Imputation)**: Managing structural zeroes and filtering null geometries.
5. **Type Casting & Downcasting (Optimization)**: Memory optimization for large-scale spatial grids.
6. **One-Hot Encoding**: Machine-readable categorical features.
7. **Data Splitting**: 60% Train, 20% Val, 20% Test splits.

In [8]:
import pandas as pd
import numpy as np
import requests
import time
import datetime
import pytz
from requests.exceptions import RequestException
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

## 1. Multi-Source Ingestion & API Polling
Instead of passing a URL directly to pandas, we write a robust extraction function. This function handles API timeouts, implements rate limiting (to avoid getting IP blocked), and uses exponential backoff if the server goes down.

In [9]:
def fetch_usgs_data_resiliently(start_date, end_date, min_mag=3.0, max_retries=3):
    """Fetches USGS data with rate limiting, timeouts, and exponential backoff."""
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime={start_date}&endtime={end_date}&minmagnitude={min_mag}"
    
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1}: Fetching data from USGS...")
            # Add a 1-second delay for rate limiting
            time.sleep(1) 
            response = requests.get(url, timeout=10)
            response.raise_for_status() # Raise exception for 4xx/5xx HTTP errors
            
            # If successful, parse the CSV response directly
            from io import StringIO
            csv_data = StringIO(response.text)
            df = pd.read_csv(csv_data)
            print(f"Successfully fetched {len(df)} records.")
            return df
            
        except RequestException as e:
            wait_time = 2 ** attempt # Exponential backoff: 1s, 2s, 4s...
            print(f"API request failed: {e}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            
    print("Max retries reached. Data extraction failed.")
    return pd.DataFrame() # Return empty DF so kernel doesn't crash

# Calculate dates for the last 6 months
end_time = datetime.datetime.now()
start_time = end_time - datetime.timedelta(days=180)

raw_df = fetch_usgs_data_resiliently(start_time.strftime('%Y-%m-%d'), end_time.strftime('%Y-%m-%d'))
raw_df.head()

Attempt 1: Fetching data from USGS...
Successfully fetched 9768 records.


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-06-11T23:39:21.740Z,19.1566,-65.297600,8.000,4.02,md,29.0,219.0,0.8492,0.38,...,2026-06-12T01:49:59.556Z,"94 km N of Culebra, Puerto Rico",earthquake,1.56,2.500,0.120000,23.0,reviewed,pr,pr
1,2026-06-11T21:32:47.364Z,-18.9647,169.053200,168.418,5.00,mb,46.0,86.0,3.9290,0.97,...,2026-06-11T21:49:04.040Z,"68 km NNW of Isangel, Vanuatu",earthquake,10.29,8.259,0.080000,51.0,reviewed,us,us
2,2026-06-11T20:37:29.510Z,18.8595,-64.667333,59.470,3.05,md,7.0,253.0,0.4393,0.05,...,2026-06-11T20:57:17.700Z,"59 km NNE of Cruz Bay, U.S. Virgin Islands",earthquake,2.16,2.190,0.178965,3.0,reviewed,pr,pr
3,2026-06-11T19:30:08.540Z,34.2386,25.123500,30.561,4.50,mb,70.0,81.0,1.0640,0.72,...,2026-06-11T20:24:12.040Z,"85 km S of Pýrgos, Greece",earthquake,7.08,6.227,0.064000,71.0,reviewed,us,us
4,2026-06-11T18:07:55.289Z,-27.8085,-71.502000,18.800,4.40,mwr,29.0,170.0,0.8850,0.54,...,2026-06-11T18:27:17.040Z,"112 km NW of Vallenar, Chile",earthquake,5.57,5.543,0.052000,36.0,reviewed,us,us


## 2. Temporal Normalization
Time-series forecasting requires strict temporal consistency. We will standardize our timestamps to Coordinated Universal Time (UTC) and create a "bucketed" time column (flooring events to the nearest hour) to create uniform intervals.

In [10]:
df = raw_df.copy()

# Ensure time is a datetime object explicitly bound to UTC
df['time'] = pd.to_datetime(df['time'], utc=True)

# Uniform Time-Aggregation: Bucket events to the nearest Hour
df['time_bucket_1H'] = df['time'].dt.floor('h')

# Uniform Time-Aggregation: Bucket events to the nearest Day
df['time_bucket_1D'] = df['time'].dt.floor('d')

df[['time', 'time_bucket_1H', 'time_bucket_1D']].head()

,time,time_bucket_1H,time_bucket_1D
0,2026-06-11 23:39:21.740000+00:00,2026-06-11 23:00:00+00:00,2026-06-11 00:00:00+00:00
1,2026-06-11 21:32:47.364000+00:00,2026-06-11 21:00:00+00:00,2026-06-11 00:00:00+00:00
2,2026-06-11 20:37:29.510000+00:00,2026-06-11 20:00:00+00:00,2026-06-11 00:00:00+00:00
3,2026-06-11 19:30:08.540000+00:00,2026-06-11 19:00:00+00:00,2026-06-11 00:00:00+00:00
4,2026-06-11 18:07:55.289000+00:00,2026-06-11 18:00:00+00:00,2026-06-11 00:00:00+00:00


## 3. Spatial Deduplication & Coordinate Validation
Corrupt data is common in live APIs. We enforce strict boundary filtering for Lat/Lon and remove spatio-temporal duplicates (multiple sensor pings for the exact same event location and time).

In [11]:
initial_len = len(df)

# 1. Coordinate Validation (Boundary Filtering)
df = df[(df['latitude'] >= -90.0) & (df['latitude'] <= 90.0)]
df = df[(df['longitude'] >= -180.0) & (df['longitude'] <= 180.0)]

# 2. Spatial Deduplication
# If two events happen at the exact same lat, lon, and time_bucket_1H, it might be a duplicate sensor read.
df = df.drop_duplicates(subset=['latitude', 'longitude', 'time_bucket_1H'], keep='first')

print(f"Dropped {initial_len - len(df)} corrupt or duplicate records.")

Dropped 0 corrupt or duplicate records.


## 4. Handling Missing Data (Imputation)
Missing coordinates make the data spatially useless, so those must be dropped. Missing metric values (like depth) can be filled.

In [12]:
# Filtering Null Geometries
# We already filtered out-of-bound coords, but we must explicitly drop nulls
df.dropna(subset=['latitude', 'longitude'], inplace=True)

# Handling structural zeroes in numerical metrics
# For earthquake depth/magnitude, a null might imply a shallow event or unrecorded magnitude.
# We will impute missing depths with the median depth of the dataset.
median_depth = df['depth'].median()
df['depth'] = df['depth'].fillna(median_depth)
df['mag'] = df['mag'].fillna(0.0) # Structural zero for missing magnitude

print(f"Null geometries remaining: {df[['latitude', 'longitude']].isnull().sum().sum()}")

Null geometries remaining: 0


## 5. Type Casting & Downcasting (Optimization)
Python default `Float64` wastes immense memory. We will cast coordinates and metrics to `Float32`, and convert repeating text (like the seismic network or magnitude type) into `Categorical` types to drastically reduce RAM usage.

In [13]:
# Check memory usage before optimization
mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage BEFORE optimization: {mem_before:.2f} MB")

# Downcasting floats
float_cols = ['latitude', 'longitude', 'depth', 'mag']
df[float_cols] = df[float_cols].astype('float32')

# Categorizing strings
categorical_cols = ['net', 'magType', 'type', 'status']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Check memory usage after optimization
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage AFTER optimization: {mem_after:.2f} MB")
if mem_before > 0:
    print(f"Memory reduced by {((mem_before - mem_after) / mem_before) * 100:.1f}%")

df.info()

Memory usage BEFORE optimization: 6.10 MB
Memory usage AFTER optimization: 3.96 MB
Memory reduced by 35.1%
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9768 entries, 0 to 9767
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   time             9768 non-null   datetime64[ns, UTC]
 1   latitude         9768 non-null   float32            
 2   longitude        9768 non-null   float32            
 3   depth            9768 non-null   float32            
 4   mag              9768 non-null   float32            
 5   magType          9768 non-null   category           
 6   nst              9767 non-null   float64            
 7   gap              9767 non-null   float64            
 8   dmin             9764 non-null   float64            
 9   rms              9767 non-null   float64            
 10  net              9768 non-null   category           
 11  id               9768 non-n

## 6. One-Hot Encoding
Before saving and feeding to ML models, we must One-Hot Encode categorical columns so they are machine-readable.

In [14]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cols_to_encode = [col for col in categorical_cols if col in df.columns]

if cols_to_encode:
    encoded_data = encoder.fit_transform(df[cols_to_encode])
    encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(cols_to_encode), index=df.index)
    
    # Cast generated One-Hot columns to float32 to conserve memory
    encoded_df = encoded_df.astype('float32')
    
    # Append to dataframe
    df = pd.concat([df, encoded_df], axis=1)
    print(f"Added {len(encoded_df.columns)} One-Hot Encoded features.")
    print(df.head())

Added 26 One-Hot Encoded features.
                              time   latitude   longitude       depth   mag  \
0 2026-06-11 23:39:21.740000+00:00  19.156601  -65.297600    8.000000  4.02   
1 2026-06-11 21:32:47.364000+00:00 -18.964701  169.053207  168.417999  5.00   
2 2026-06-11 20:37:29.510000+00:00  18.859501  -64.667336   59.470001  3.05   
3 2026-06-11 19:30:08.540000+00:00  34.238602   25.123501   30.561001  4.50   
4 2026-06-11 18:07:55.289000+00:00 -27.808500  -71.501999   18.799999  4.40   

  magType   nst    gap    dmin   rms  ... magType_md magType_ml magType_mw  \
0      md  29.0  219.0  0.8492  0.38  ...        1.0        0.0        0.0   
1      mb  46.0   86.0  3.9290  0.97  ...        0.0        0.0        0.0   
2      md   7.0  253.0  0.4393  0.05  ...        1.0        0.0        0.0   
3      mb  70.0   81.0  1.0640  0.72  ...        0.0        0.0        0.0   
4     mwr  29.0  170.0  0.8850  0.54  ...        0.0        0.0        0.0   

  magType_mwb magType

## 7. Data Splitting (Train, Validation, Test)
We divide our dataset into Training (60%), Cross-Validation (20%), and Testing (20%) datasets.

In [15]:
# First, split into Train (60%) and Temp (40% for Val and Test)
train_df, temp_df = train_test_split(df, test_size=0.40, random_state=42)

# Next, split the Temp data equally into Validation (20%) and Test (20%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Training Data Shape: {train_df.shape} (60%)")
print(f"Validation Data Shape: {val_df.shape} (20%)")
print(f"Testing Data Shape: {test_df.shape} (20%)")

Training Data Shape: (5860, 50) (60%)
Validation Data Shape: (1954, 50) (20%)
Testing Data Shape: (1954, 50) (20%)


## 8. Saving Final Processed & Split Data
We save the Master dataset (for spatial binning/dashboarding) alongside the individual ML splits.

In [16]:
# Save optimized master dataset (Used for step 2: Hexagonal Binning)
df.to_csv("optimized_earthquake_data_master.csv", index=False)

# Save ML Splits
train_df.to_csv("train_earthquake_data.csv", index=False)
val_df.to_csv("val_earthquake_data.csv", index=False)
test_df.to_csv("test_earthquake_data.csv", index=False)

print("Professional data ingestion pipeline completed. Master file and splits saved successfully.")

Professional data ingestion pipeline completed. Master file and splits saved successfully.
